# EvidenceAI — Stance Classifier Fine-Tuning (SciFact)

**What this notebook does:** fine-tunes a 3-class sentence-pair classifier
(claim, evidence passage) → SUPPORT / CONTRADICT / NEUTRAL, using the SciFact
triples prepared locally by `notebooks/prepare_scifact.py`
(`data/scifact/{train,validation,test}.jsonl`).

**Why:** `backend/app/pipeline/stance.py` currently uses zero-shot NLI
(`facebook/bart-large-mnli`), which measurably under-detects CONTRADICT on
the real EvidenceAI corpus — general-domain MNLI isn't calibrated for
hedged/statistical scientific skepticism language (e.g. "no significant
difference" reads as NEUTRAL instead of CONTRADICT). This notebook produces
a checkpoint for `STANCE_MODEL_PATH`; `stance.py` is already label-name-mapped
(reads `model.config.id2label` by string, not by index), so the fine-tuned
model this notebook produces drops in with **zero code changes** — this
notebook sets `id2label`/`label2id` to the exact same SUPPORT/CONTRADICT/
NEUTRAL vocabulary for that reason.

**Data note:** SciFact's official `test` split ships with zero evidence
annotations (it's the held-out shared-task leaderboard set — verified live
while building `prepare_scifact.py`). This notebook does not treat it as
usable. Instead: SciFact's `train.jsonl` is split further (stratified) into
a training set and a held-out test set never seen during training or
epoch-level model selection; SciFact's `validation.jsonl` is used, as
intended, for epoch-level validation. See the markdown cell before the
train/test split for the full reasoning.

**Runtime estimate (free T4):** the prepared dataset is small (~1,290
training triples after held-out-test carve-out). At an effective batch size
of 32 (16 × 2 grad-accum steps) and 4 epochs, that's roughly 160 optimizer
steps. Expect on the order of **10–20 minutes end-to-end** on a free T4,
including model/tokenizer download and dependency install — most of that is
one-time overhead, not the training loop itself. This has been checked for
correctness against each library's documented API, but **has not been
executed against a live Colab GPU runtime** (this notebook was authored in a
sandboxed environment with no GPU/Colab access) — if a library's API has
moved since this was written, the most likely failure points are called out
in comments near `TrainingArguments` and the tokenizer loading cell, both of
which include a defensive fallback.

**Before running:** upload `data/scifact/train.jsonl` and
`data/scifact/validation.jsonl` (produced locally by `prepare_scifact.py`)
to your Google Drive at `MyDrive/EvidenceAI/data/scifact/` — see the data
loading cell for the exact expected path and an alternative upload-widget
method if you'd rather not pre-sync Drive.


## 1. Mount Google Drive (first, before anything else)

Free Colab sessions can disconnect at any time — all checkpoints, logs, and the final model are written to Drive, never to local Colab storage, so a disconnect doesn't lose the run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os

# Everything this notebook writes lives under here. Change PROJECT_DIR if you
# use a different folder name in your Drive.
PROJECT_DIR = '/content/drive/MyDrive/EvidenceAI'
DATA_DIR = f'{PROJECT_DIR}/data/scifact'
RUN_DIR = f'{PROJECT_DIR}/stance_training'
CHECKPOINT_DIR = f'{RUN_DIR}/checkpoints'
FINAL_MODEL_DIR = f'{RUN_DIR}/final_model'
CONFUSION_MATRIX_DIR = f'{RUN_DIR}/eval_plots'

for d in (RUN_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, CONFUSION_MATRIX_DIR):
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Writing all outputs under:', RUN_DIR)


## 2. Install pinned dependencies, confirm GPU

Pinned to versions known to work together for this task. If Colab's preinstalled `transformers` is newer and this conflicts, restart the runtime after this cell (Colab sometimes needs a restart to pick up a downgraded/upgraded core package).

In [ ]:
%%capture
!pip install -q \
    transformers==4.46.3 \
    datasets==2.20.0 \
    accelerate==0.33.0 \
    evaluate==0.4.2 \
    scikit-learn==1.5.1


In [ ]:
import torch
import transformers
import datasets as hf_datasets

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', hf_datasets.__version__)
print()

if torch.cuda.is_available():
    device = torch.device('cuda')
    props = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('No GPU detected — go to Runtime > Change runtime type > T4 GPU, then re-run.')


## 3. Load the prepared SciFact JSONL splits from Drive

Expects `train.jsonl` and `validation.jsonl` at `MyDrive/EvidenceAI/data/scifact/` (upload them there first — drag into the Drive web UI, or use `google.colab.files.upload()` once to seed the folder, see the commented-out alternative below).

`test.jsonl` is loaded too, purely to confirm it's still empty as expected (SciFact's official held-out set has no evidence annotations at all) — it is not used for anything past this cell.

In [ ]:
import json

# Alternative if you haven't synced Drive yet — uncomment to upload interactively,
# then move the files into DATA_DIR:
# from google.colab import files
# uploaded = files.upload()  # select train.jsonl and validation.jsonl

def load_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_raw = load_jsonl(f'{DATA_DIR}/train.jsonl')
validation_raw = load_jsonl(f'{DATA_DIR}/validation.jsonl')
test_raw = load_jsonl(f'{DATA_DIR}/test.jsonl')

print(f'train.jsonl:      {len(train_raw)} rows')
print(f'validation.jsonl: {len(validation_raw)} rows')
print(f'test.jsonl:       {len(test_raw)} rows (expected 0 — see note above)')

assert train_raw, (
    f'No training data found at {DATA_DIR}/train.jsonl — upload it to Drive first.'
)


## 4. Build the actual train / validation / held-out-test split

SciFact's official `test` split can't be used as a held-out test set — it has
zero evidence annotations (verified while building `prepare_scifact.py`).
Using `validation.jsonl` for *both* epoch-level model selection (best-checkpoint
picking) *and* the final "how good is this model really" evaluation would be
optimistically biased — we'd be reporting performance on the exact data used
to choose the checkpoint.

So the split used here is:
- **`train`**: 90% of `train.jsonl`, stratified by label — what the model
  actually trains on.
- **`validation`**: `validation.jsonl` as-is, used only for `eval_strategy="epoch"`
  / best-checkpoint selection during training (never trained on).
- **`held_out_test`**: the other 10% of `train.jsonl`, carved out *before*
  training starts and never touched until the final evaluation cell — this is
  the true blind test set, standing in for SciFact's unusable official one.

The exact `held_out_test` row indices (into `train.jsonl`) are saved to Drive
right after the split (see the code cell below) so `compare_classifiers.py`
can load the identical test set locally instead of re-deriving the split —
re-derivation would silently diverge if `scikit-learn`'s version or shuffle
algorithm ever differs between this Colab environment and the local venv,
and the baseline and fine-tuned numbers must be measured on the same rows to
be comparable.


In [ ]:
from sklearn.model_selection import train_test_split

labels_for_split = [row['label'] for row in train_raw]
indices_for_split = list(range(len(train_raw)))

train_split, held_out_test_split, train_indices, held_out_test_indices = train_test_split(
    train_raw,
    indices_for_split,
    test_size=0.10,
    random_state=42,
    stratify=labels_for_split,
)

print(f'train (fitting):        {len(train_split)}')
print(f'validation (epoch eval): {len(validation_raw)}')
print(f'held_out_test (final):  {len(held_out_test_split)}')

from collections import Counter

for name, rows in [('train', train_split), ('validation', validation_raw), ('held_out_test', held_out_test_split)]:
    counts = Counter(r['label'] for r in rows)
    print(f'{name} label counts: {dict(counts)}')

# Save the exact held-out-test indices (into train.jsonl's row order) to Drive
# — see the markdown cell above for why this needs to be an explicit saved
# artifact rather than something the comparison script re-derives.
split_indices_path = f'{RUN_DIR}/scifact_split_indices.json'
with open(split_indices_path, 'w', encoding='utf-8') as f:
    json.dump({
        'source_file': 'train.jsonl',
        'split_params': {'test_size': 0.10, 'random_state': 42, 'stratify': 'label'},
        'train_indices': train_indices,
        'held_out_test_indices': held_out_test_indices,
    }, f, indent=2)
print('Saved split indices to', split_indices_path)


## 5. Load tokenizer + model — DeBERTa-v3-base primary, RoBERTa-base fallback

`MODEL_NAME` is the one variable to change to swap architectures. `id2label`/`label2id` use the exact SUPPORT/CONTRADICT/NEUTRAL vocabulary `stance.py` already expects by name.

In [ ]:
PRIMARY_MODEL_NAME = 'microsoft/deberta-v3-base'
FALLBACK_MODEL_NAME = 'roberta-base'

id2label = {0: 'SUPPORT', 1: 'CONTRADICT', 2: 'NEUTRAL'}
label2id = {v: k for k, v in id2label.items()}

from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = PRIMARY_MODEL_NAME
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f'Loaded tokenizer for {MODEL_NAME}')
except Exception as exc:
    print(f'Tokenizer load failed for {MODEL_NAME} ({exc!r}) — falling back to {FALLBACK_MODEL_NAME}')
    MODEL_NAME = FALLBACK_MODEL_NAME
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Using MODEL_NAME =', MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
print(model.config.id2label)


## 6. Tokenize as sentence pairs (claim, passage)

`truncation='only_second'` keeps the claim intact and truncates the (usually longer) passage side if the pair exceeds `max_length=256` — the claim is short and always fully informative; the passage is where truncation should land if anything has to give.

In [ ]:
from datasets import Dataset, DatasetDict

MAX_LENGTH = 256


def to_hf_dataset(rows):
    return Dataset.from_list([
        {'claim': r['claim'], 'passage': r['passage'], 'label': label2id[r['label']]}
        for r in rows
    ])


raw_datasets = DatasetDict({
    'train': to_hf_dataset(train_split),
    'validation': to_hf_dataset(validation_raw),
    'held_out_test': to_hf_dataset(held_out_test_split),
})


def tokenize_batch(batch):
    return tokenizer(
        batch['claim'],
        batch['passage'],
        truncation='only_second',
        max_length=MAX_LENGTH,
    )


tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=['claim', 'passage'],
)

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_datasets


### Final split sizes and per-class counts (paste into the report)

Same numbers as step 4's split cell, reprinted here in a fixed-width table
right after tokenization — this is the "final" shape of what's actually fed
to the `Trainer` (tokenization doesn't change row counts, only adds
`input_ids`/`attention_mask` columns).

In [ ]:
from collections import Counter

_splits_for_report = [
    ('train', train_split),
    ('validation', validation_raw),
    ('held_out_test', held_out_test_split),
]

print(f"{'split':<14}{'size':>8}{'SUPPORT':>10}{'CONTRADICT':>13}{'NEUTRAL':>10}")
for name, rows in _splits_for_report:
    counts = Counter(r['label'] for r in rows)
    print(
        f"{name:<14}{len(rows):>8}"
        f"{counts.get('SUPPORT', 0):>10}"
        f"{counts.get('CONTRADICT', 0):>13}"
        f"{counts.get('NEUTRAL', 0):>10}"
    )


## 7. Class weights for the loss

Measured distribution in `train.jsonl` (from `prepare_scifact.py`, combined
train+validation): SUPPORT 47.8%, CONTRADICT 26.6%, NEUTRAL 25.5%. This is a
**mild** imbalance, not the severe (<15% CONTRADICT) case `prepare_scifact.py`
was checking for — no automatic mitigation fired there.

It's also milder than earlier framing here assumed. The measured zero-shot
baseline (see README: Model Provenance) shows **SUPPORT recall (29.0%) is
actually worse than CONTRADICT recall (52.9%)** — the baseline over-predicts
NEUTRAL generally, it doesn't specifically under-detect CONTRADICT. Tuning
aggressive inverse-frequency weighting to fix "the CONTRADICT problem" would
have been solving a problem the data doesn't show.

Given that, `WEIGHTING_MODE` (next cell) defaults to `'none'` — a mild real
imbalance plus a failure mode that measurement showed to be different than
assumed is exactly the case where unweighted loss is the right starting
point. Two escalation paths are available if a class visibly collapses once
you see the first epoch's confusion matrix (see the runbook for what
"collapse" looks like in practice): `'sqrt'` (dampened inverse-frequency —
try this first) and `'inverse'` (the original full "balanced" formula this
notebook used before this revision).


In [ ]:
import numpy as np

# 'none'   -> unweighted cross-entropy (try this first — see markdown above)
# 'sqrt'   -> dampened inverse-frequency (first escalation if a class collapses)
# 'inverse'-> full "balanced" inverse-frequency (this notebook's old default)
WEIGHTING_MODE = 'none'

train_labels = np.array(tokenized_datasets['train']['label'])
class_counts = np.array([np.sum(train_labels == i) for i in range(3)])
n_samples = len(train_labels)
n_classes = 3

if WEIGHTING_MODE == 'none':
    class_weights = np.ones(n_classes)
elif WEIGHTING_MODE == 'sqrt':
    class_weights = np.sqrt(n_samples / (n_classes * class_counts))
elif WEIGHTING_MODE == 'inverse':
    class_weights = n_samples / (n_classes * class_counts)
else:
    raise ValueError(f"Unknown WEIGHTING_MODE: {WEIGHTING_MODE!r} — expected 'none', 'sqrt', or 'inverse'")

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f'WEIGHTING_MODE = {WEIGHTING_MODE!r}')
for i, name in id2label.items():
    print(f'{name:<12} count={class_counts[i]:<5} weight={class_weights[i]:.3f}')


In [ ]:
from transformers import Trainer


class WeightedLossTrainer(Trainer):
    """Trainer with class-weighted cross-entropy — see the markdown cell above
    for why weighting was chosen over oversampling."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


## 8. Metrics: overall accuracy, macro F1, per-class P/R/F1, confusion matrix

`metric_for_best_model="macro_f1"` (next cell) — **not** a single class's
recall. Measurement disproved the assumption this notebook originally used to
justify selecting on `contradict_recall`: the zero-shot baseline's real
weakness is SUPPORT recall (29.0%), which is *worse* than its CONTRADICT
recall (52.9%), and it over-predicts NEUTRAL generally rather than
specifically failing on CONTRADICT. Selecting the best checkpoint on any one
class's recall would optimize for a failure mode the data doesn't actually
show. Macro F1 averages per-class F1 equally, so it can't be dominated by the
majority classes (like accuracy can) and can't overfit to one class's recall
at the expense of the other two. `compute_metrics` still computes and
returns per-class recall/precision/F1 (including `contradict_recall`) for
diagnostic visibility in the report — only checkpoint *selection* changed.


In [ ]:
import evaluate
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

LABEL_NAMES = [id2label[i] for i in range(3)]
CONTRADICT_IDX = label2id['CONTRADICT']


def print_confusion_matrix(labels, predictions):
    cm = confusion_matrix(labels, predictions, labels=list(range(3)))
    header = '           ' + ''.join(f'{n:>12}' for n in LABEL_NAMES)
    print(header)
    for i, row in enumerate(cm):
        print(f'{LABEL_NAMES[i]:>10} ' + ''.join(f'{v:>12}' for v in row))
    return cm


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_metric.compute(predictions=predictions, references=labels, average='macro')['f1']
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, labels=list(range(3)), zero_division=0
    )

    print('\nEval confusion matrix:')
    print_confusion_matrix(labels, predictions)
    print(classification_report(labels, predictions, target_names=LABEL_NAMES, zero_division=0))

    metrics = {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'contradict_recall': recall[CONTRADICT_IDX],
        'contradict_precision': precision[CONTRADICT_IDX],
        'contradict_f1': f1[CONTRADICT_IDX],
    }
    for i, name in id2label.items():
        metrics[f'{name.lower()}_precision'] = precision[i]
        metrics[f'{name.lower()}_recall'] = recall[i]
        metrics[f'{name.lower()}_f1'] = f1[i]
    return metrics


## 9. TrainingArguments (tuned for a free T4) + Trainer

`transformers` renamed `evaluation_strategy` to `eval_strategy` around
v4.46 (pinned above) — the try/except below falls back to the old name if
Colab ends up on an older `transformers` than pinned, so this cell works
either way without manual edits.


In [ ]:
from transformers import TrainingArguments

training_args_kwargs = dict(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    save_strategy='epoch',
    load_best_model_at_end=True,
    # macro_f1, not a single class's recall: the measured zero-shot baseline
    # showed SUPPORT recall (29.0%) is worse than CONTRADICT recall (52.9%)
    # and that the model over-predicts NEUTRAL generally — selecting on one
    # class's recall (the notebook's original 'contradict_recall') would have
    # optimized for a failure mode the data doesn't actually show. See the
    # markdown cell above for the full reasoning.
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=10,
    report_to='none',
)

try:
    training_args = TrainingArguments(eval_strategy='epoch', **training_args_kwargs)
except TypeError:
    print('This transformers version expects evaluation_strategy, not eval_strategy — falling back.')
    training_args = TrainingArguments(evaluation_strategy='epoch', **training_args_kwargs)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## 10. Train

Checkpoints save to Drive after every epoch (`save_strategy="epoch"`, `output_dir` under `RUN_DIR`) — safe to resume from Drive if the session disconnects mid-run.

In [ ]:
train_result = trainer.train()
print(train_result)


## 11. Final evaluation on the held-out test split

This is `held_out_test` (carved out of `train.jsonl` in step 4) — never seen
during training or during epoch-level checkpoint selection, so this is an
honest final number, unlike re-reporting `validation` performance.


In [ ]:
test_predictions = trainer.predict(tokenized_datasets['held_out_test'])
test_metrics = test_predictions.metrics
print('Held-out test metrics:')
for k, v in test_metrics.items():
    print(f'  {k}: {v}')


In [ ]:
import matplotlib.pyplot as plt

final_preds = np.argmax(test_predictions.predictions, axis=-1)
final_labels = test_predictions.label_ids
cm = confusion_matrix(final_labels, final_preds, labels=list(range(3)))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(LABEL_NAMES)
ax.set_yticklabels(LABEL_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Held-out test confusion matrix')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im)
fig.tight_layout()

cm_path = f'{CONFUSION_MATRIX_DIR}/held_out_test_confusion_matrix.png'
fig.savefig(cm_path, dpi=150)
plt.show()
print('Saved confusion matrix to', cm_path)


## 12. Save the model + tokenizer to Drive

This is what `STANCE_MODEL_PATH` in `backend/app/config.py` should point at — `stance.py` loads it via `AutoModelForSequenceClassification.from_pretrained(...)`/`AutoTokenizer.from_pretrained(...)`, so a plain `save_pretrained()` output directory is exactly what it expects.

In [ ]:
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

with open(f'{RUN_DIR}/held_out_test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(test_metrics, f, indent=2)

print('Saved model + tokenizer to', FINAL_MODEL_DIR)
print('Saved held-out test metrics to', f'{RUN_DIR}/held_out_test_metrics.json')


## 13. Package the model for download

Zips `FINAL_MODEL_DIR` to Drive, and offers a direct browser download via `files.download()` as well, in case you'd rather grab it straight from this session than dig through Drive.

In [ ]:
import shutil

zip_base_path = f'{RUN_DIR}/stance_model'  # shutil appends .zip
zip_path = shutil.make_archive(zip_base_path, 'zip', FINAL_MODEL_DIR)

print('Model directory (Drive):', FINAL_MODEL_DIR)
print('Zipped model (Drive):   ', zip_path)

from google.colab import files

files.download(zip_path)
